# Notebook 3 — Kafka Streaming (Producer & Consumer)
**Project**: Real-Time Retail Analytics & Demand Prediction Platform  
**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar  
**Stack**: Apache Kafka 3.7 (KRaft mode) | kafka:9092  

Creates Kafka topic `retail-txn-v2`, streams records at 300 rec/sec, and verifies consumption.

---

## 3.1 Config

In [1]:
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Kafka — KRaft mode, no Zookeeper
KAFKA_BROKERS = 'kafka:9092'
TOPIC         = 'retail-txn-v2'       # new topic, won't conflict with old one
STREAM_RATE   = 2000                   # records per second
BATCH_SIZE    = 500

# Dataset
SCALED_PATH   = '/home/jovyan/work/retail_20m.parquet'

# How many records to stream in this demo run
# (streaming all 15M would take ~14 hours at 300/sec — we stream a representative batch)
MAX_RECORDS   = 500_000

print('Config loaded.')
print(f'Topic: {TOPIC}')
print(f'Rate:  {STREAM_RATE} rec/sec')
print(f'Max:   {MAX_RECORDS:,} records')

Config loaded.
Topic: retail-txn-v2
Rate:  2000 rec/sec
Max:   500,000 records


## 3.2 Create Kafka Topic

In [2]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(
    bootstrap_servers=KAFKA_BROKERS,
    client_id='retail-v2-admin',
    request_timeout_ms=15000
)

try:
    admin.create_topics([
        NewTopic(name=TOPIC, num_partitions=3, replication_factor=1)
    ])
    print(f'✅ Topic \'{TOPIC}\' created (3 partitions).')
except TopicAlreadyExistsError:
    print(f'Topic \'{TOPIC}\' already exists — OK.')

# List all topics
topics = admin.list_topics()
print(f'\nAll Kafka topics: {topics}')
admin.close()

Topic 'retail-txn-v2' already exists — OK.

All Kafka topics: ['retail_transactions', 'retail-transactions', 'retail-txn-v2', '__consumer_offsets']


## 3.3 Load Data Sample for Streaming

In [3]:
import pyarrow.parquet as pq

# Read only first few row groups (not entire 15M) to stay within RAM
pf = pq.ParquetFile(SCALED_PATH)
print(f'Total row groups: {pf.metadata.num_row_groups}')

# Read first row group — should have enough rows
df_stream = pf.read_row_group(0).to_pandas()

# If not enough, read one more
if len(df_stream) < MAX_RECORDS and pf.metadata.num_row_groups > 1:
    df_stream = pd.concat([
        df_stream,
        pf.read_row_group(1).to_pandas()
    ], ignore_index=True)

df_stream = df_stream.head(MAX_RECORDS)
print(f'Records to stream: {len(df_stream):,}')
df_stream.head(3)

Total row groups: 19
Records to stream: 500,000


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 11:45:00,6.973993,13085,United Kingdom,83.687916
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 14:45:00,6.738989,13085,United Kingdom,80.867865
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.431585,13085,United Kingdom,77.179023


## 3.4 Kafka Producer — Stream at Controlled Rate

In [4]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKERS,
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
    key_serializer=lambda k: str(k).encode('utf-8'),
    linger_ms=10,
    batch_size=2000,
    compression_type='gzip'
)

sent  = 0
start = time.time()

print(f'Streaming {len(df_stream):,} records to Kafka topic \'{TOPIC}\' at ~{STREAM_RATE} rec/s...\n')

for i in range(0, len(df_stream), BATCH_SIZE):
    batch = df_stream.iloc[i : i + BATCH_SIZE]
    
    for _, row in batch.iterrows():
        msg = row.to_dict()
        producer.send(TOPIC, key=row['StockCode'], value=msg)
        sent += 1
    
    producer.flush()
    
    # Rate control
    elapsed  = time.time() - start
    expected = sent / STREAM_RATE
    if expected > elapsed:
        time.sleep(expected - elapsed)
    
    if sent % 50_000 == 0:
        rate = sent / (time.time() - start)
        print(f'  Sent {sent:>8,} | {rate:>6.0f} rec/s | Elapsed: {time.time()-start:>6.1f}s')

producer.close()
total_time = time.time() - start
print(f'\n✅ Streaming complete!')
print(f'   Records sent : {sent:,}')
print(f'   Total time   : {total_time:.1f}s')
print(f'   Avg rate     : {sent/total_time:.0f} rec/s')

Streaming 500,000 records to Kafka topic 'retail-txn-v2' at ~2000 rec/s...

  Sent   50,000 |   2000 rec/s | Elapsed:   25.0s
  Sent  100,000 |   2000 rec/s | Elapsed:   50.0s
  Sent  150,000 |   2000 rec/s | Elapsed:   75.0s
  Sent  200,000 |   2000 rec/s | Elapsed:  100.0s
  Sent  250,000 |   2000 rec/s | Elapsed:  125.0s
  Sent  300,000 |   2000 rec/s | Elapsed:  150.0s
  Sent  350,000 |   2000 rec/s | Elapsed:  175.0s
  Sent  400,000 |   2000 rec/s | Elapsed:  200.0s
  Sent  450,000 |   2000 rec/s | Elapsed:  225.0s
  Sent  500,000 |   2000 rec/s | Elapsed:  250.0s

✅ Streaming complete!
   Records sent : 500,000
   Total time   : 250.0s
   Avg rate     : 2000 rec/s


## 3.5 Kafka Consumer — Verify Messages

In [5]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKERS,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=10000,
    group_id='verify-v2'
)

samples = []
count   = 0
for msg in consumer:
    count += 1
    if len(samples) < 5:
        samples.append(msg.value)
    if count >= 10000:  # count first 10k then stop
        break

consumer.close()

print(f'Consumer read {count:,} messages from topic \'{TOPIC}\'.')
print(f'\nSample messages:')
pd.DataFrame(samples)

Consumer read 10,000 messages from topic 'retail-txn-v2'.

Sample messages:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.431585,13085,United Kingdom,77.179023
1,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 20:45:00,1.245220,13085,United Kingdom,29.885271
2,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 10:46:00,3.896287,13085,United Kingdom,46.755441
3,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 19:06:00,5.971226,13078,United Kingdom,59.712265
4,489436,22119,PEACE WOODEN BLOCK LETTERS,3,2009-12-01 14:06:00,6.808051,13078,United Kingdom,20.424152


## 3.6 Topic Partition Stats

In [6]:
from kafka import KafkaConsumer

c = KafkaConsumer(
    bootstrap_servers=KAFKA_BROKERS,
    group_id='stats-v2'
)

partitions = c.partitions_for_topic(TOPIC)
if partitions:
    print(f'Topic: {TOPIC}')
    print(f'Partitions: {len(partitions)}')
    for p in sorted(partitions):
        print(f'  Partition {p}: active')
else:
    print(f'Topic {TOPIC} not found.')

c.close()

Topic: retail-txn-v2
Partitions: 3
  Partition 0: active
  Partition 1: active
  Partition 2: active


## 3.7 Summary

| Step | Status |
|------|--------|
| Create Kafka topic `retail-txn-v2` | ✅ |
| Stream 500K records at ~300 rec/s | ✅ |
| Consumer verification | ✅ |
| Partition stats | ✅ |

**Kafka Config:**
- Mode: KRaft (no Zookeeper)
- Broker: kafka:9092
- Partitions: 3
- Compression: gzip

**Next**: Notebook 4 — Read from Kafka, write to Delta Lake on MinIO (`retail-v2/`)